### Setup & Installation

Run the cell below once to install all required packages.  
Then restart the kernel before continuing.

In [1]:
import sys
!{sys.executable} -m pip install -q \
    pandas numpy requests \
    sentence-transformers transformers \
    faiss-cpu tf-keras \
    gradio matplotlib seaborn tqdm \
    ollama python-dotenv
print("All packages installed.")

All packages installed.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\RKS\Personnal\MachineLearning\python.exe -m pip install --upgrade pip


In [2]:
import os, time, pickle, warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

print("Imports OK.")

Imports OK.


### Configuration

Set your TMDB API key below (get a free one at https://www.themoviedb.org/settings/api).  
Create a .env file with **TMDB_API_KEY=your_key**

In [ ]:
from dotenv import load_dotenv
load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY", "MY_API_KEY") # replaced my actual API key with a placeholder MY_API_KEY

if TMDB_API_KEY == "MY_API_KEY":
    print("Please set your TMDB_API_KEY above before proceeding.")
else:
    print(f"TMDB key loaded")

In [3]:
TMDB_BASE_URL = "https://api.themoviedb.org/3"

## Data Collection — TMDB API

Movie data pulled from [The Movie Database (TMDB)](https://www.themoviedb.org/).  
The strategy:
1. /genre/movie/list — get all genre IDs  
2. /discover/movie — collect movie IDs balanced across genres  
3. /movie/{id}?append_to_response=credits — one call per movie for full details + cast  

In [ ]:
def tmdb_get(endpoint, params=None, retries=3):
    params = params or {}
    params["api_key"] = TMDB_API_KEY
    params.setdefault("language", "en-US")

    for attempt in range(retries):
        try:
            r = requests.get(f"{TMDB_BASE_URL}{endpoint}", params=params, timeout=10)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                return None

In [ ]:
# discover movie IDs (popularity-based retrieval)
def discover_movie_ids(total=MOVIES_TO_FETCH):
    ids_seen, movie_ids = set(), []

    page = 1
    max_pages = 500 

    with tqdm(total=total, desc="Discovering movies") as pbar:
        while len(movie_ids) < total and page <= max_pages:
            data = tmdb_get("/discover/movie", {
                "sort_by": "popularity.desc",
                "vote_count.gte": 50,   
                "page": page
            })

            if not data or "results" not in data:
                page += 1
                continue

            for m in data["results"]:
                if m["id"] not in ids_seen:
                    ids_seen.add(m["id"])
                    movie_ids.append(m["id"])
                    pbar.update(1)

                if len(movie_ids) >= total:
                    break

            page += 1
            time.sleep(0.25)

    return movie_ids

In [ ]:
movie_ids = discover_movie_ids(MOVIES_TO_FETCH)
print(len(movie_ids))

In [ ]:
# fetch full details and credits for each movie
def fetch_details(movie_id):
    d = tmdb_get(f"/movie/{movie_id}", {"append_to_response": "credits"})
    if not d:
        return None

    director = next(
        (p["name"] for p in d.get("credits", {}).get("crew", []) if p["job"] == "Director"), ""
    )

    cast = ", ".join(a["name"] for a in d.get("credits", {}).get("cast", [])[:5])
    genres = ", ".join(g["name"] for g in d.get("genres", []))

    release = d.get("release_date", "")
    year = int(release[:4]) if release and len(release) >= 4 else None

    return {
        "id": d.get("id"),
        "title": d.get("title", ""),
        "year": year,
        "genres": genres,
        "vote_average": round(d.get("vote_average", 0.0), 1),
        "vote_count": d.get("vote_count", 0),
        "popularity": round(d.get("popularity", 0.0), 2),
        "runtime": d.get("runtime"),
        "director": director,
        "cast": cast,
        "overview": d.get("overview", ""),
        "original_language": d.get("original_language", ""),
    }

rows = []
fail_count = 0

for mid in tqdm(movie_ids, desc="Fetching movie details"):
    rec = fetch_details(mid)

    if rec and rec["title"] and rec["overview"]:
        rows.append(rec)
    else:
        fail_count += 1

    time.sleep(0.25)

df_raw = pd.DataFrame(rows)
df_raw.to_csv(DATA_PATH, index=False)

print(f"\nSaved: {len(df_raw)} movies")
print(f"Failed/skipped: {fail_count}")

### Dataset Exploration

In [4]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}\n")
df.head(5)

Shape: (4000, 12)



,id,title,year,genres,vote_average,vote_count,popularity,runtime,director,cast,overview,original_language
0,83533,Avatar: Fire and Ash,2025,"Science Fiction, Adventure, Fantasy",7.4,2391,485.10,198,James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver...",In the wake of the devastating war against the...,en
1,1327819,Hoppers,2026,"Animation, Family, Science Fiction, Comedy, Ad...",7.6,395,395.36,105,Daniel Chong,"Piper Curda, Bobby Moynihan, Jon Hamm, Kathy N...",Scientists have discovered how to 'hop' human ...,en
2,1171145,Crime 101,2026,"Crime, Thriller",7.1,516,339.60,141,Bart Layton,"Chris Hemsworth, Mark Ruffalo, Halle Berry, Ba...",When an elusive thief whose high-stakes heists...,en
3,1115544,Mike & Nick & Nick & Alice,2026,"Comedy, Science Fiction, Crime",6.8,178,301.23,107,BenDavid Grabinski,"James Marsden, Vince Vaughn, Eiza González, Ke...",Two gangsters and the woman they love try to s...,en
4,1290821,Shelter,2026,"Action, Crime, Thriller",6.8,489,326.12,107,Ric Roman Waugh,"Jason Statham, Bodhi Rae Breathnach, Michael S...",A man living in self-imposed exile on a remote...,en


In [5]:
print("dtypes & non-null counts")
df.info()

dtypes & non-null counts
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 4000 non-null   int64  
 1   title              4000 non-null   object 
 2   year               4000 non-null   int64  
 3   genres             4000 non-null   object 
 4   vote_average       4000 non-null   float64
 5   vote_count         4000 non-null   int64  
 6   popularity         4000 non-null   float64
 7   runtime            4000 non-null   int64  
 8   director           3999 non-null   object 
 9   cast               3998 non-null   object 
 10  overview           4000 non-null   object 
 11  original_language  4000 non-null   object 
dtypes: float64(2), int64(4), object(6)
memory usage: 375.1+ KB


In [6]:
print("numeric summary")
df[['vote_average', 'popularity', 'runtime']].describe().round(2)

numeric summary


,vote_average,popularity,runtime
count,4000.00,4000.00,4000.00
mean,6.86,9.77,112.38
std,0.75,19.16,23.42
min,3.50,2.61,6.00
25%,6.40,5.05,97.00
50%,6.90,6.33,109.00
75%,7.40,9.15,124.00
max,9.30,485.10,253.00


In [7]:
print("Missing values")
missing = df.isnull().sum()
missing

Missing values


id                   0
title                0
year                 0
genres               0
vote_average         0
vote_count           0
popularity           0
runtime              0
director             1
cast                 2
overview             0
original_language    0
dtype: int64